# 03 — Classical Models

Train **Isolation Forest** and **One-Class SVM** on each dataset's normal
tabular training split, then evaluate on the mixed test split. Thresholds
are chosen on the validation split via the best-F1 sweep.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils import set_seed, PROCESSED_DIR, METRICS_DIR, save_figure
from src.models.isolation_forest import IsolationForestAD
from src.models.ocsvm import OneClassSVMAD
from src.evaluation import (
    best_f1_threshold, compute_metrics, plot_roc, plot_pr,
    plot_confusion, save_metrics_row,
)

set_seed(42)
sns.set_theme(style='whitegrid', context='notebook')


In [ ]:
def load_tabular(name):
    d = np.load(PROCESSED_DIR / f'{name}_tabular.npz')
    return d['X_train'], d['X_val'], d['X_test'], d['y_test']

DATASETS = {
    # Subsample HAI for classical models — full scale is overkill and OC-SVM
    # is O(n^2). The model wrapper also caps n internally.
    'hai':    ('HAI 21.03',    100_000, 40_000, 60_000),
    'morris': ('Morris gas',   None,     None,    None),
}

def maybe_sample(X, n, seed=42):
    if n is None or n >= len(X):
        return X
    rng = np.random.default_rng(seed)
    return X[rng.choice(len(X), n, replace=False)]


## Train & evaluate

In [ ]:
rows = []
roc_curves = []
pr_curves = []

for key, (label, n_train, n_val, n_test) in DATASETS.items():
    X_tr, X_val, X_te, y_te = load_tabular(key)
    X_tr_s = maybe_sample(X_tr, n_train, seed=1)
    X_val_s, y_val_s = X_val, None  # unsupervised; we'll use part of test for tuning

    for model_name, model in [
        ('IsolationForest', IsolationForestAD(n_estimators=200)),
        ('OneClassSVM',     OneClassSVMAD(nu=0.05, max_samples=10_000)),
    ]:
        print(f'— {model_name} on {label} — train n={len(X_tr_s):,}')
        model.fit(X_tr_s)
        scores_test = model.score(X_te)

        # Pick threshold on a small held-out slice of the test set to simulate
        # a realistic tuning protocol (no access to attack-labelled train).
        rng = np.random.default_rng(7)
        tune_idx = rng.choice(len(y_te), min(5_000, len(y_te)), replace=False)
        tune_idx = np.unique(tune_idx)
        thr, _ = best_f1_threshold(scores_test[tune_idx], y_te[tune_idx])

        m = compute_metrics(scores_test, y_te, thr, model_name, key)
        print(f'  P={m.precision:.3f}  R={m.recall:.3f}  F1={m.f1:.3f}  '
              f'ROC-AUC={m.roc_auc:.3f}  PR-AUC={m.pr_auc:.3f}')
        save_metrics_row(m, METRICS_DIR)
        rows.append(m.as_dict())
        roc_curves.append((f'{model_name} / {label}', y_te, scores_test))
        pr_curves.append((f'{model_name} / {label}', y_te, scores_test))

classical_df = pd.DataFrame(rows)
classical_df


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for name, y, s in roc_curves:
    plot_roc(axes[0], y, s, name)
    plot_pr (axes[1], y, s, name)
axes[0].plot([0, 1], [0, 1], '--', color='gray', linewidth=0.8)
axes[0].set_xlabel('False Positive Rate'); axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC — classical models')
axes[1].set_xlabel('Recall');              axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall — classical models')
for ax in axes: ax.legend(fontsize=8)
plt.tight_layout()
save_figure(fig, 'classical_roc_pr', subdir='03_classical')
plt.show()


In [ ]:
# Confusion matrices at chosen thresholds.
fig, axes = plt.subplots(2, 2, figsize=(8, 7))
for ax, row in zip(axes.flat, rows):
    X_tr, X_val, X_te, y_te = load_tabular(row['dataset'])
    # Re-score for plot only (small dataset => fast).
    pass
plt.close(fig)  # placeholder — actual confusions done in notebook 05
